# Example 4.6 / 1: Collections of Collections

Companion notebook for **4.6 Collections of Sets** in the AMPL lab Documentation, section "Collections of Collections". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Collections of Collections

Collections can be nested. Every campaign uses some channels, and every campaign and channel
pair has its own creative variants. The second indexing phrase must use `CHANNELS[campaign]`:
writing `channel in CHANNELS` is invalid because `CHANNELS` names the collection, not one of its
member sets, and `campaign` must come first, as Indexing Expressions (AMPL lab documentation, section 4.3) explains.

From Python, a nested collection takes a dictionary whose keys are the parent tuples, here
`(campaign, channel)` pairs. A data file gives each member set with its subscripts.
`union {c in CAMPAIGNS} CHANNELS[c]` combines the member sets of a collection:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CAMPAIGNS;                         # marketing campaigns
set CHANNELS {CAMPAIGNS};              # channels each campaign uses
set CREATIVES {campaign in CAMPAIGNS,  # creatives for each campaign and its channels
               channel in CHANNELS[campaign]};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["CAMPAIGNS"] = ["spring", "fall"]
ampl.set["CHANNELS"] = {"spring": ["web", "tv"], "fall": ["web"]}
ampl.set["CREATIVES"] = {("spring", "web"): ["banner", "video"],
                         ("spring", "tv"): ["spot30"],
                         ("fall", "web"): ["banner"]}

In [5]:
# run.py
print(ampl.set["CREATIVES"]["spring", "web"].to_list())
print(ampl.get_value("card(CREATIVES['spring', 'tv'])"))
print(ampl.get_data("union {c in CAMPAIGNS} CHANNELS[c]").to_list())

['banner', 'video']
1
['web', 'tv']


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set CAMPAIGNS := spring fall;
set CHANNELS[spring] := web tv;
set CHANNELS[fall] := web;
set CREATIVES[spring,web] := banner video;
set CREATIVES[spring,tv] := spot30;
set CREATIVES[fall,web] := banner;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display CREATIVES;
display union {c in CAMPAIGNS} CHANNELS[c];

set CREATIVES[spring,web] := banner video;
set CREATIVES[spring,tv] := spot30;
set CREATIVES[fall,web] := banner;

set  union {c in CAMPAIGNS}  CHANNELS[c] := web tv;

